# Consolidate and clean QCOR inspections

1. Set a survey ID to summarize the deficiencies by survey 
2. Create a variable to identify the kind of survey by matching to CMS3070G dates / ccns. 
3. Add the translations of the deficiencies into the table
4. Add a `nyt_special` category to the deficiency table to identify areas of particular interest
5. Create a  summary tables by survey, which can be used later on to combine them. 

## Combine batches & set survey id

Each batch that was parsed was saved into its own RDS file in the `qcor_tables` folder. This combines them into one dataframe, and creates a survey identifier based on ccn, survey date, and survey type.

There is a possibility that we combine surveys of the same type on the same day, but I'm not sure that's possible. There's really no other way to set the unique identifier, since the header is the only thing that separates them. 

In [1]:
pacman::p_load("tidyverse", "lubridate", "janitor", "here", "glue")
i_am("qcor/06-save-survey-results.ipynb")

here() starts at /Users/sarah/Github/nc-homesdisabled


In [2]:

# now put them all together into one data frame
# THIS HAS TO BE CHANGED AS SHOWN HERE AT SOME POINT TO GET A SURVEY ID. I DON'T KNOW HOW TO MAKE SURE IT IS THE SAME EACH TIME..

qcor_scrape <-
  list.files(
    here("qcor/qcor_tables"),
    pattern = "\\.rds$",
    full.names = TRUE
  ) |>
  map(readRDS) |>
  list_rbind() |>
  rowid_to_column() |>
  mutate(
    survey_date = str_extract(header, "\\d{2}/\\d{2}/\\d{4}") |> mdy(),
    survey_type = str_extract(header, "(SPECIAL|COMPLAINT|STANDARD)"),
  ) |>
  arrange(ccn, survey_date, rowid) |>
  group_by(ccn, header) |>
  mutate(
    survey_id = paste(
      ccn,
      format(survey_date, "%Y%m%d"),
      survey_type , 
      sep = "_"
    )
  ) |>
  ungroup() |> 
  select ( survey_id, rowid, ccn, survey_type, survey_date, level, tag_number, tag_group =  deficiency_description, followup_date_1, followup_date_2) |> 
  # add a counter to see how many rows are in this survey as deficiencies. There will always be at least 1, which will say , "None"
  group_by ( survey_id )    |> 
  mutate ( survey_row =  row_number()  ) |>
  ungroup()   |> 
  arrange ( rowid)



glimpse(qcor_scrape)

Rows: 427,392
Columns: 11
$ survey_id       <chr> "01G011_20230713_STANDARD", "01G011_20220810_STANDARD", "01G011_20210319_STANDARD", "01G011_20210319_ST…
$ rowid           <int> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, …
$ ccn             <chr> "01G011", "01G011", "01G011", "01G011", "01G011", "01G011", "01G011", "01G011", "01G011", "01G011", "01…
$ survey_type     <chr> "STANDARD", "STANDARD", "STANDARD", "STANDARD", "STANDARD", "SPECIAL", "STANDARD", "STANDARD", "STANDAR…
$ survey_date     <date> 2023-07-13, 2022-08-10, 2021-03-19, 2021-03-19, 2021-03-19, 2021-03-19, 2019-06-12, 2018-06-13, 2017-0…
$ level           <chr> "Standard", "Standard", "Standard", "Standard", "Standard", "None", "Standard", "None", "Standard", "St…
$ tag_number      <chr> "W0252", "W0331", "W0336", "W0341", "W0371", "None", "W0341", "None", "W0159", "W0249", "W0257", "W0322…
$ tag_group       <chr> "PROGRAM DOCUMENTATION", "NURSING SERVICES", "N

### Which surveys are the annual vs other standard ones? 

The CMS 3070G data shows the dates of the recertification surveys. Use that list and apply it to the scraped data. 

In [3]:
cms3070_recert_dates <- 
  readRDS(here("cms_3070g/cms3070g_allfields.rds")) |> 
  # just in case there are duplicates, but there aren't right now
  distinct ( ccn = provider_ccn, survey_date) |> 
  mutate ( nyt_survey_type = "Recertification", 
          survey_type = "STANDARD") 

In [4]:
qcor_survey_types <- 
  qcor_scrape |> 
  left_join ( cms3070_recert_dates, join_by ( ccn, survey_date, survey_type)) |> 
  mutate ( nyt_survey_type = if_else ( is.na (nyt_survey_type ), str_to_title(survey_type), nyt_survey_type) ) |> 
  relocate ( nyt_survey_type , .after=ccn)

glimpse ( qcor_survey_types |>  sample_n ( 20))

Rows: 20
Columns: 12
$ survey_id       <chr> "42G145_20101124_STANDARD", "15G636_20190201_STANDARD", "42G136_20140516_STANDARD", "33G826_20191122_ST…
$ rowid           <int> 295537, 141159, 295123, 219473, 372679, 63098, 424145, 27206, 92449, 53887, 102693, 61146, 238007, 2790…
$ ccn             <chr> "42G145", "15G636", "42G136", "33G826", "50G007", "09G075", "55G679", "05G494", "14G208", "05G959", "14…
$ nyt_survey_type <chr> "Standard", "Recertification", "Standard", "Recertification", "Complaint", "Complaint", "Complaint", "S…
$ survey_type     <chr> "STANDARD", "STANDARD", "STANDARD", "STANDARD", "COMPLAINT", "COMPLAINT", "COMPLAINT", "STANDARD", "STA…
$ survey_date     <date> 2010-11-24, 2019-02-01, 2014-05-16, 2019-11-22, 2020-12-03, 2014-07-14, 2018-06-27, 2012-06-06, 2012-0…
$ level           <chr> "Standard", "Standard", "Standard", "Standard", "None", "None", "None", "Standard", "Standard", "Standa…
$ tag_number      <chr> "W0268", "W0130", "W0455", "E0004", "None", "None", 

## Apply more detailed code translations

The scraped code translations are sometimes a little unclear, but it's better than having nothing. This especially happens when there are continuations of the regulations that have several categories of compliance. 


In [5]:

# to remove the regulation number from the description, and put it in its own area
# can't use a variable in the same row normally, so have to do this weird thing. 
# fixed() keeps the parens in the original and instead of seeing them as part of the regex.

str_remove_rowwise <- function ( string, pattern) {
    map2_chr ( string, pattern, \(s,p) {
            if (is.na(s) || is.na(p))  return(s)
            str_remove (s, fixed(p)) |>  str_remove("^[.\\s]+") |>  str_trunc (100)
            }
        )  
}


This also cleans up the translations just a little -- it truncates the descriptions to 150 characters, removes the Code of Federal Regulations code, and adds a tag for emergency preparedness deficiencies.

In [6]:


translations <- readRDS(here("qcor/data/qcor_deficiency_codes.RDS"))|> 
    select ( tag_number, tag_group = deficiency_description, deficiency_level= level, w_code, us_code, code_description = description, code_guidance=guidance) |> 
    # the emergency codes aren't in the translations.
    mutate ( 
      short_description = if_else (str_starts (tag_number, "E") , "Emergency Preparedness", str_remove_rowwise ( code_description, us_code) ) |> 
                          str_trim () |>  str_squish() |> 
                          str_trunc ( width=150)
    ) |> 
    group_by ( tag_number) |> 
    slice_tail (n=1) |> 
    ungroup() 
    



#### Special types of deficiencies

Create a column to flag special types of deficiencies, i.e., related to abuse and neglect or staffing inadequacies. This is a very preliminary list, but we can revisit it later. 


In [7]:
abuse_neglect_codes <- paste0("W0", c(127, 128, 150, 151, 153, 154, 155, 156, 157, 285, 287, 286, 292, 298, 306, 310))
staffing_codes <- paste0("W0", c( 158 , 167, 170, 485, 344, 301, 183, 184, 185, 186, 187))

translations_to_apply <- 
  translations |> 
  mutate ( nyt_special = case_when ( tag_number %in% abuse_neglect_codes ~ "Abuse / neglect", 
                                      tag_number %in% staffing_codes ~ "Staffing", 
                                    .default = NA_character_
                                  )
          ) |> 
  select  ( tag_number, short_description, nyt_special, us_code) 
            



#### Apply translations and save

(A few details about the provider is also added here)




In [10]:
latest_provider_types <- readRDS(here("qcor/data/qcor_provider_master.RDS")) |> 
                    select ( ccn, provider_name, city, state, certified_beds, ownership_type, as_of_year = year) 


In [11]:
qcor_survey_details <- 
  qcor_survey_types |> 
  left_join ( translations_to_apply,   join_by( tag_number) ) |> 
  left_join ( latest_provider_types, join_by ( ccn))

  

In [12]:
# uncomment to resave

saveRDS(qcor_survey_details, here("qcor/data/qcor_survey_details.RDS"))

## Summarize surveys by provider

Provide some key statistics on each survey. These will be used later on to summarize providers. 





In [13]:
survey_summaries <- 
  qcor_survey_details |> 
  group_by ( survey_id ) |> 
  mutate  ( 
              deficiency_ct = sum ( level != "None", na.rm=TRUE ) , 
              condition = sum ( level == "Condition" ) ,
              abuse_neglect = sum ( nyt_special == "Abuse / neglect", na.rm=TRUE),
              staffing = sum ( nyt_special == "Staffing", na.rm=TRUE)
  ) |> 
  slice_head (n = 1) |> 
  ungroup()  |> 
  select ( survey_id:nyt_survey_type, 
           survey_date, deficiency_ct:staffing,
          provider_name, city, state , 
          ownership_type, ownership_year = as_of_year)
  

# uncomment to resave
saveRDS(survey_summaries , here("qcor/data/qcor_survey_summaries.RDS"))